In [1]:
import chromadb
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

/home/mark/Downloads/llm_agent_engineer_assessment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CHROMA_DB_PATH = "chroma_db"

In [3]:
client = chromadb.PersistentClient(path=CHROMA_DB_PATH)

In [5]:
collection_name = "task_2_1"
collection = client.get_or_create_collection(collection_name)

In [7]:
collection.peek()

{'ids': [],
 'embeddings': array([], dtype=float64),
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents', 'embeddings'],
 'data': None,
 'metadatas': []}

In [12]:
def fixed_length_chunking(text, chunk_size=100, overlap=20):
    """
    Splits text into fixed-length chunks by word count, with optional overlap.    
    Parameters:
    text (str): The input text to chunk.
    chunk_size (int): Number of words per chunk.
    overlap (int): Number of overlapping words between
    consecutive chunks.
    Returns:
    List[str]: List of word chunks.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [ ]:
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct-AWQ",device_map="cuda",torch_dtype=torch.float16) 
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct-AWQ") 

In [9]:
with open("supplementary_files/context.txt") as f:
    context_text = f.read()

In [13]:
chunks = fixed_length_chunking(context_text,20,4)

In [14]:
ids = [f"id_{i}" for i in range(len(chunks))]

In [16]:
collection.add(
    ids=ids,
    documents=chunks
)

## Task 2.2

In [23]:
all_quest =["What are the main challenges in renewable energy storage technologies?","How do governments influence the adoption of renewable energy?",
            "Why do some communities oppose large-scale wind farms?"]

for user_query in all_quest:
    context = collection.query(
        query_texts=[user_query],
        n_results=5
    )['documents'][0]

    prompt = f"{user_query}. Use this as context for answering: {context}. Keep it short and simple."
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs,max_new_tokens=8000)
    print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    print("")
        

The main challenges in renewable energy storage technologies include:

1. **Reliability**: Ensuring that batteries or other storage systems can generate enough power when needed, especially during low-wind conditions.
2. **Scalability**: Making sure these storage systems can handle large amounts of energy, which is crucial for supporting consistent power generation.
3. **Affordability**: Reducing the cost of storage solutions so they can be widely adopted and integrated into existing grids.
4. **Environmental Impact**: Developing sustainable and eco-friendly materials for storage components.
5. **Energy Density**: Increasing the amount of energy stored per unit volume or mass to improve efficiency.
6. **Interoperability**: Ensuring different storage technologies can work together seamlessly with renewable energy sources and existing infrastructure.<|im_end|>

Governments influence the adoption of renewable energy through various means:

1. **Subsidies**: Providing financial support hel

- Model giving output that are not found in the provided text. Example: The last point of the first respond. 
- Overall cover all info from the passage.

## Task 2.3

Specific ask model do not include information that outside the provided text.

In [24]:
all_quest =["What are the main challenges in renewable energy storage technologies?","How do governments influence the adoption of renewable energy?",
            "Why do some communities oppose large-scale wind farms?"]

for user_query in all_quest:
    context = collection.query(
        query_texts=[user_query],
        n_results=5
    )['documents'][0]

    prompt = f"{user_query}. Use this as context for answering: {context}. Keep it short and simple. Do not include information that are not included in provided context."
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs,max_new_tokens=8000)
    print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    print("")
        

The main challenges in renewable energy storage technologies include:

1. **Reliability**: Ensuring that energy storage systems can generate megawatts of power even in low-wind conditions.
2. **Scalability**: Developing solutions that can handle large-scale energy production and distribution.
3. **Affordability**: Making storage systems cost-effective for widespread adoption.
4. **Environmental Impact**: Reducing the environmental footprint of storage materials and processes.
5. **Energy Density**: Increasing the amount of energy stored per unit volume or weight to improve efficiency.

These challenges require significant advancements in technology and policy support to enable renewable energy to compete with fossil fuels as a consistent base-load source.<|im_end|>

Governments significantly influence the adoption of renewable energy through several key policies:

1. **Subsidies**: Providing financial support helps reduce the initial cost of renewable energy systems, making them more a

- The model output is more concise especially the  last output.

## Task 2.4

In [25]:
all_quest =["What are the main challenges in renewable energy storage technologies?","How do governments influence the adoption of renewable energy?",
            "Why do some communities oppose large-scale wind farms?"]

for user_query in all_quest:
    # context = collection.query(
    #     query_texts=[user_query],
    #     n_results=5
    # )['documents'][0]

    # prompt = f"{user_query}. Use this as context for answering: {context}. Keep it short and simple."
    prompt = f"{user_query}.Keep it short and simple."
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs,max_new_tokens=8000)
    print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    print("")
        

Main challenges in renewable energy storage technologies include:
1. Cost: High initial costs for storage systems.
2. Efficiency: Energy loss during conversion and storage processes.
3. Capacity: Insufficient storage capacity to match peak demand.
4. Durability: Long-term reliability and maintenance requirements.
5. Environmental impact: Production and disposal of materials used in storage systems.<|im_end|>

Governments can influence renewable energy adoption through policies like tax incentives, subsidies, and mandates for renewable energy usage. They also invest in research and development to improve technologies, and set standards for energy efficiency. These actions create demand and reduce costs, making renewables more attractive to investors and consumers.<|im_end|>

Some communities oppose large-scale wind farms because they may view them as visual or noise disruptions, or because they could affect local wildlife, leading to concerns about aesthetics and environmental impact.<|

- More or less same output with Task 2.2 but not much detail.

In [27]:
all_quest =["What are the main challenges in renewable energy storage technologies?","How do governments influence the adoption of renewable energy?",
            "Why do some communities oppose large-scale wind farms?"]

for user_query in all_quest:
    context = collection.query(
        query_texts=[user_query],
        n_results=5
    )['documents'][0]

    prompt = f"{user_query}. Use this as context for answering: {context}. Keep it short and simple."
    # prompt = f"{user_query}.Keep it short and simple."
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs,max_new_tokens=100)
    print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    print("")
        

The main challenges in renewable energy storage technologies include:

1. **Reliability**: Ensuring that energy can be stored and released efficiently and consistently.
2. **Scalability**: Developing large-scale storage systems that can handle high power demands without losing efficiency or capacity.
3. **Cost**: Making storage systems affordable so they can compete with traditional energy sources.
4. **Technology Limitations**: Improving battery life, charging speed, and reducing costs.
5. **Environmental Impact**: Minimizing the

Governments influence the adoption of renewable energy through various policies:

1. **Subsidies and Incentives**: Governments provide financial support to encourage investment in renewable energy projects.
2. **Penalties for Carbon Emissions**: By imposing taxes or penalties on fossil fuels, governments make renewables more attractive economically.
3. **Investment in Grid Infrastructure**: Building robust, efficient energy grids helps integrate renewable so

In [30]:
all_quest =["What are the main challenges in renewable energy storage technologies?","How do governments influence the adoption of renewable energy?",
            "Why do some communities oppose large-scale wind farms?","How to farm strawberry?"]

for user_query in all_quest:
    context = collection.query(
        query_texts=[user_query],
        n_results=5
    )['documents'][0]

    prompt = f"{user_query}. Use this as context for answering: {context}. Keep it short and simple. Make sure user query is relevant to the context. If the answer cannot be found in the provided context, state that explicitly and do not use outside knowledge."
    # prompt = f"{user_query}.Keep it short and simple."
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(**inputs,max_new_tokens=8000)
    print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    print("")
        

The main challenges in renewable energy storage technologies include:

1. Developing reliable and scalable storage solutions to store energy produced by intermittent sources like solar and wind.
2. Ensuring energy can be generated at consistent levels even in low-wind conditions.
3. Overcoming high initial costs to make storage systems affordable.
4. Enhancing battery efficiency and reducing costs.
5. Integrating storage systems into existing energy grids efficiently and cost-effectively.<|im_end|>

Governments influence the adoption of renewable energy through various policies. They provide subsidies for green technologies, which makes these options more affordable. This helps reduce costs and increases adoption. Additionally, governments impose penalties for carbon emissions and invest in grid infrastructure to support renewable energy systems. These actions help overcome initial high costs and address concerns like aesthetics and impacts on local wildlife. Through education and comm